In [1]:
import pandas as pd 

df = pd.read_csv('../fixed_final_data_product.csv')

In [2]:
# Count missing values per row
df["missing_count"] = df.isna().sum(axis=1)

# Count how many rows have each missing_count value
unique_missing_counts = (
    df["missing_count"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "Missing Columns", "missing_count": "Number of Missing Columns"})
)

# Format nicely
print("=== Missing Data Summary (per row) ===")
print(unique_missing_counts.to_markdown(index=False))

# remove rows with more than 172 missing columns
df_cleaned = df[df["missing_count"] <= 172].drop(columns=["missing_count"])


=== Missing Data Summary (per row) ===
|   Number of Missing Columns |   count |
|----------------------------:|--------:|
|                           0 |     829 |
|                          22 |      52 |
|                          42 |      14 |
|                          44 |       5 |
|                          64 |       1 |
|                          66 |       1 |
|                         172 |     130 |


In [3]:
from sklearn.impute import SimpleImputer

# the only non-numeric column is 'lap_id', so we can focus on numeric columns only

# Select numeric columns
num_cols = df.select_dtypes(include=["number"]).columns

# Find columns that have missing values
missing_cols = [col for col in num_cols if df[col].isna().any()]

# Compute skewness for those columns
skewness_values = df[missing_cols].skew().to_dict()

# Split columns based on skewness
# use median for skewed data (|skewness| >= 0.5), mean for symmetric
median_cols = [col for col, skew in skewness_values.items() if abs(skew) >= 0.5]
mean_cols   = [col for col, skew in skewness_values.items() if abs(skew) < 0.5]

# Apply imputers
median_imputer = SimpleImputer(strategy="median")
mean_imputer   = SimpleImputer(strategy="mean")

if median_cols:
    df[median_cols] = median_imputer.fit_transform(df[median_cols])

if mean_cols:
    df[mean_cols] = mean_imputer.fit_transform(df[mean_cols])

# total number of missing values in the entire dataset
total_missing = df.isna().sum().sum()
print(f"Total missing values in dataset: {total_missing}")

Total missing values in dataset: 0


In [7]:
import pandas as pd

# Define all prefixes you care about
prefixes = ["BPS_", "BPE_", "THS_", "THE_", "STS_", "STM_", "STE_", "APX1_", "APX2_"]

# Create a results dictionary
prefix_summary = {}

for prefix in prefixes:
    cols = [c for c in df.columns if c.startswith(prefix)]
    prefix_summary[prefix] = cols

# Optional: Convert to a dataframe summary
summary_df = pd.DataFrame({
    "Prefix": prefix_summary.keys(),
    "Column_Count": [len(v) for v in prefix_summary.values()],
    "Columns": [", ".join(v) for v in prefix_summary.values()]
})

print("\n=== Summary Table ===")
display(summary_df)



=== Summary Table ===


,Prefix,Column_Count,Columns
0,BPS_,22,"BPS_SPEED, BPS_THROTTLE, BPS_STEER, BPS_BRAKE,..."
1,BPE_,22,"BPE_SPEED, BPE_THROTTLE, BPE_STEER, BPE_BRAKE,..."
2,THS_,22,"THS_SPEED, THS_THROTTLE, THS_STEER, THS_BRAKE,..."
3,THE_,22,"THE_SPEED, THE_THROTTLE, THE_STEER, THE_BRAKE,..."
4,STS_,22,"STS_SPEED, STS_THROTTLE, STS_STEER, STS_BRAKE,..."
5,STM_,20,"STM_SPEED, STM_THROTTLE, STM_STEER, STM_BRAKE,..."
6,STE_,22,"STE_SPEED, STE_THROTTLE, STE_STEER, STE_BRAKE,..."
7,APX1_,20,"APX1_SPEED, APX1_THROTTLE, APX1_STEER, APX1_BR..."
8,APX2_,20,"APX2_SPEED, APX2_THROTTLE, APX2_STEER, APX2_BR..."
